# B-bot stub: спреды LA + маркеры сделок (как гир 1.0)

Локальный Plotly-график по тикам коллектора D и журналу заглушек B-bot.

Данные уже лежат на Mac (не VPS):

| Что | Путь |
|---|---|
| Тики LA (lean, спреды считаются при чтении) | `output/bbot_plot/la_ticks.parquet` |
| Журнал dual-leg | `output/bbot_plot/journal/event_date=*/legs.jsonl` |

Окно: **2026-08-17 15:15 UTC → 2026-08-18 10:00 UTC** (прогон stub, `signal_test`, LA).

Ось Y — **спред %**, как в `plot_strategy` гира 1.0. `fill_price` журнала — L1, на график не идёт.

Это не PnL и не доказательство, что бот и коллектор видели один L1 (разные WS). Маркеры сажаются на ряд D по `signal_ts_ms` / `fill_ts_ms`.


## 1. Конфиг


In [29]:
from pathlib import Path
import sys

REPO = Path.cwd()
if not (REPO / "research" / "plot_bbot_spreads.py").exists():
    REPO = Path("/Users/mishatrubik/Desktop/spread")
sys.path.insert(0, str(REPO))

from research.plot_bbot_spreads import load_bbot_plot_inputs, plot_strategy

COIN = "LA"
START = "2026-08-17"  # UTC, inclusive
END = "2026-08-18"    # UTC, inclusive
TICKS_PATH = REPO / "output" / "bbot_plot" / "la_ticks.parquet"
JOURNAL_DIR = REPO / "output" / "bbot_plot" / "journal"
OUT_HTML = REPO / "output" / "bbot_plot" / "bbot_LA_spreads.html"

MAX_POINTS = 4000         # прореживание линий; 0 = все тики
MARKER_MODE = "both"       # fill | signal | both
AVG_WINDOW_SEC = 2.0       # Gate B; None чтобы выключить
MAX_LATENCY_OKX_MS = 40.0
MAX_LATENCY_BYBIT_MS = 25.0
WIDTH, HEIGHT = 1400, 700

print("repo", REPO)
print("ticks", TICKS_PATH, "exists", TICKS_PATH.exists())
print("journal", JOURNAL_DIR, "exists", JOURNAL_DIR.exists())


repo /Users/mishatrubik/Desktop/spread
ticks /Users/mishatrubik/Desktop/spread/output/bbot_plot/la_ticks.parquet exists True
journal /Users/mishatrubik/Desktop/spread/output/bbot_plot/journal exists True


## 2. Загрузка тиков и сделок


In [30]:
ticks, trades, intents, notes, meta = load_bbot_plot_inputs(
    TICKS_PATH,
    JOURNAL_DIR,
    coin=COIN,
    start=START,
    end=END,
)
miss = [n for n in notes if n.startswith("fill_lookup_miss")]
print(meta)
print("fill_lookup_hit", meta["n_intents"] - len(miss), "miss", len(miss))
for note in notes[:12]:
    print("note:", note)


{'n_ticks': 175526, 'n_intents': 33, 'n_trades': 17, 'n_closed': 16, 'n_eod_open': 1, 'tick_span': ('2026-08-17T15:14:55.629000+00:00', '2026-08-18T09:58:53.269000+00:00')}
fill_lookup_hit 33 miss 0


## 3. График как в гире 1.0

`spread_long` (синий), `spread_short` (красный), пунктир Gate B MA.
Маркеры: треугольник вверх = open fill, вниз = close fill, кружок = signal, ромб = открытая позиция на конце ряда.


In [31]:
period = START if START == END else f"{START}..{END}"
fig = plot_strategy(
    ticks,
    trades,
    title=(
        f"{COIN} {period} — D ticks + bbot stub "
        f"(closed={meta['n_closed']} eod_open={meta['n_eod_open']} marker={MARKER_MODE})"
    ),
    width=WIDTH,
    height=HEIGHT,
    max_points=MAX_POINTS,
    marker_mode=MARKER_MODE,
    avg_window_sec=AVG_WINDOW_SEC,
    max_latency_okx_ms=MAX_LATENCY_OKX_MS,
    max_latency_bybit_ms=MAX_LATENCY_BYBIT_MS,
)
fig.show()
OUT_HTML.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(str(OUT_HTML), include_plotlyjs=True, full_html=True)
print("wrote", OUT_HTML)


gate_b_ma ticks=175526 window_sec=2.0


wrote /Users/mishatrubik/Desktop/spread/output/bbot_plot/bbot_LA_spreads.html
